<a href="https://colab.research.google.com/github/erickvaldezsallagos/ingenieria/blob/main/Basededatos(Samsung).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Scraper del catálogo de samsung.com/mx (para Google Colab)
#   1) Se lee robots.txt y se respetan sus reglas (y el Crawl-delay, si existe)
#   2) Se usa el sitemap del sitio si lo publica; si no, se siguen los enlaces desde la portada
#   3) Se descarga en paralelo, con pausas, reintentos y un tope de bloqueos
#   4) Se guarda un CSV con hasta META_PRODUCTOS productos y se descarga
# ============================================================

import gzip
import json
import re
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ---------- Configuración ----------
INICIO = "https://www.samsung.com/mx/"
HOSTS = ("www.samsung.com", "samsung.com")
PREFIJO = "/mx/"         # solo México
META_PRODUCTOS = 6000      # número de productos a reunir
MAX_PAGINAS = 30000        # tope de seguridad de páginas descargadas
HILOS = 6                  # descargas simultáneas (más de 10 aumenta el riesgo de bloqueo)
PAUSA = 0.5                # segundos de espera antes de cada petición
LOTE = 80                  # URLs por ronda
MAX_BLOQUEOS = 30          # respuestas 403 máximas antes de detenerse
ARCHIVO = "samsung_mx_productos.csv"
AGENTE = "EstudioSamsungMX/1.0"

# Páginas descartadas (imágenes, soporte, carrito, ofertas...)
NO_INTERESAN = re.compile(
    r"\.(jpg|jpeg|png|gif|svg|webp|pdf|zip|css|js|mp4|xml|json)$"
    r"|/(support|soporte|search|buscar|cart|carrito|login|account|cuenta|news|info|offer|offers)(/|$)",
    re.IGNORECASE,
)
# Se visitan primero las URLs que parecen de producto
PARECE_PRODUCTO = re.compile(r"/buy/?$|sm-[a-z]\d{3}", re.IGNORECASE)
# Código de modelo de Samsung, p. ej. SM-S928BLBOLTM
MODELO = re.compile(r"sm-[a-z]\d{3}\w*", re.IGNORECASE)

sesion = requests.Session()
sesion.headers.update({"User-Agent": AGENTE, "Accept-Language": "es-MX,es;q=0.9"})

# ---------- 1. robots.txt ----------
rp = RobotFileParser()
respuesta = sesion.get("https://www.samsung.com/robots.txt", timeout=15)
if respuesta.status_code == 200:
    rp.parse(respuesta.text.splitlines())
elif respuesta.status_code == 404:
    rp.parse([])               # sin reglas: todo permitido
else:
    raise SystemExit(f"No se pudo leer robots.txt (código {respuesta.status_code}). Se detiene el proceso.")
if not rp.can_fetch(AGENTE, INICIO):
    raise SystemExit("robots.txt no permite rastrear el sitio con este agente. Se detiene el proceso.")
retardo = rp.crawl_delay(AGENTE)
if retardo:
    PAUSA = max(PAUSA, float(retardo))
print("Pausa entre peticiones:", PAUSA, "s")

# ---------- 2. Cola de URLs ----------
cola_producto = deque()
cola_general = deque()
vistas = set()


def aceptable(url):
    """Se verifica que la URL sea de México, no esté descartada y que robots.txt la permita."""
    partes = urlparse(url)
    if partes.scheme not in ("http", "https") or partes.netloc not in HOSTS:
        return False
    if not partes.path.startswith(PREFIJO) or NO_INTERESAN.search(partes.path):
        return False
    return rp.can_fetch(AGENTE, url)


def encolar(url):
    """Se agrega la URL a la cola si no se había visto antes y es aceptable."""
    url = url.split("#")[0].split("?")[0]
    if url in vistas or not aceptable(url):
        return
    vistas.add(url)
    if PARECE_PRODUCTO.search(url):
        cola_producto.append(url)
    else:
        cola_general.append(url)


def urls_del_sitemap(url_mapa):
    """Se lee un sitemap (o un índice de sitemaps) y se devuelven las URLs que contiene."""
    salida = []
    pendientes = [url_mapa]
    leidos = set()
    while pendientes:
        actual = pendientes.pop()
        if actual in leidos:
            continue
        leidos.add(actual)
        try:
            r = sesion.get(actual, timeout=30)
            if r.status_code != 200:
                continue
            contenido = gzip.decompress(r.content) if actual.endswith(".gz") else r.content
        except (requests.RequestException, OSError):
            continue
        for loc in re.findall(r"<loc>\s*([^<\s]+)\s*</loc>", contenido.decode("utf-8", "ignore")):
            if loc.endswith((".xml", ".xml.gz")):
                pendientes.append(loc)   # es un sitemap dentro del índice
            else:
                salida.append(loc)
    return salida


encolar(INICIO)
for mapa in rp.site_maps() or []:
    for url in urls_del_sitemap(mapa):
        encolar(url)
print(f"Cola inicial: {len(cola_producto)} URLs de producto y {len(cola_general)} generales")

# ---------- 3. Descarga y extracción ----------
def descargar(url):
    """Se descarga una página y se reintenta si hay un error temporal. Devuelve (url, html, código)."""
    codigo = None
    for intento in range(3):
        time.sleep(PAUSA)
        try:
            respuesta = sesion.get(url, timeout=20)
        except requests.RequestException:
            codigo = "error de red"
            time.sleep(2 ** intento)
            continue
        codigo = respuesta.status_code
        if codigo == 200:
            return url, respuesta.text, 200
        if codigo in (429, 500, 502, 503, 504):
            time.sleep(2 ** (intento + 1))   # el servidor pide calma, así que se espera más
            continue
        break                                # 403, 404, etc.: reintentar no sirve
    return url, None, codigo


def aplanar(datos):
    """Se convierte el JSON-LD (dict, lista o @graph) en una lista plana de diccionarios."""
    elementos = datos if isinstance(datos, list) else [datos]
    salida = []
    for elemento in elementos:
        if isinstance(elemento, dict):
            salida.append(elemento)
            if "@graph" in elemento:
                salida.extend(aplanar(elemento["@graph"]))
    return salida


def precio_visible(sopa):
    """Plan B: se toma el primer texto con clase 'price' que tenga números (puede equivocarse)."""
    for etiqueta in sopa.select("[class*='price']"):
        texto = etiqueta.get_text(" ", strip=True)
        if re.search(r"\d", texto):
            return texto[:60]
    return ""


def extraer_producto(sopa, url):
    """Se devuelve un diccionario con los datos ampliados del producto (17 columnas), o None si no es de producto."""
    modelo = MODELO.search(url)
    registro = {
        "url": url,
        "modelo": modelo.group(0).upper() if modelo else "",
        "nombre": "",
        "marca": "",
        "sku": "",
        "gtin": "",
        "precio": "",
        "moneda": "",
        "disponibilidad": "",
        "calificacion": "",
        "resenas": "",
        # --- Columnas Nuevas Añadidas ---
        "categoria": "",
        "descripcion": "",
        "imagen": "",
        "color": "",
        "condicion": "",
        "imagen_og": "",
        "fuente": "",
    }

    def meta(nombre):
        etiqueta = sopa.find("meta", attrs={"property": nombre}) or sopa.find("meta", attrs={"name": nombre})
        return etiqueta.get("content", "").strip() if etiqueta else ""

    # 1) Datos estructurados JSON-LD (muchas tiendas los publican para los buscadores)
    producto = None
    for etiqueta in sopa.find_all("script", type="application/ld+json"):
        try:
            datos = json.loads(etiqueta.string or "")
        except ValueError:
            continue
        for objeto in aplanar(datos):
            tipo = objeto.get("@type")
            es_producto = tipo == "Product" or (isinstance(tipo, list) and "Product" in tipo)
            if producto is None and es_producto:
                producto = objeto

    if producto is not None:
        oferta = producto.get("offers", {})
        oferta = oferta[0] if isinstance(oferta, list) and oferta else oferta
        oferta = oferta if isinstance(oferta, dict) else {}
        marca = producto.get("brand", "")
        marca = marca.get("name", "") if isinstance(marca, dict) else marca
        calificacion = producto.get("aggregateRating", {})
        calificacion = calificacion if isinstance(calificacion, dict) else {}
        gtin = ""
        for clave in ("gtin13", "gtin14", "gtin12", "gtin8", "gtin"):
            if clave in producto and not gtin:
                gtin = str(producto[clave])

        # Extracción adicional para columnas nuevas
        imagen_prod = producto.get("image", "")
        if isinstance(imagen_prod, list) and imagen_prod:
            imagen_prod = imagen_prod[0]

        registro.update({
            "nombre": producto.get("name", ""),
            "marca": marca,
            "sku": producto.get("sku", ""),
            "gtin": gtin,
            "precio": oferta.get("price") or oferta.get("lowPrice", ""),
            "moneda": oferta.get("priceCurrency", ""),
            "disponibilidad": oferta.get("availability", ""),
            "calificacion": calificacion.get("ratingValue", ""),
            "resenas": calificacion.get("reviewCount", ""),
            "descripcion": producto.get("description", ""),
            "imagen": str(imagen_prod),
            "color": producto.get("color", ""),
            "condicion": oferta.get("itemCondition", ""),
            "categoria": meta("article:section") or meta("og:type"),
            "imagen_og": meta("og:image"),
            "fuente": "json-ld",
        })
    # 2) Meta-etiquetas Open Graph
    elif meta("og:type") == "product" or meta("product:price:amount"):
        registro.update({
            "nombre": meta("og:title"),
            "precio": meta("product:price:amount") or meta("og:price:amount"),
            "moneda": meta("product:price:currency") or meta("og:price:currency"),
            "descripcion": meta("og:description") or meta("description"),
            "categoria": meta("product:category") or meta("og:type"),
            "imagen_og": meta("og:image"),
            "fuente": "meta",
        })
    # 3) Plan B: título y precio tomados del HTML, solo si la URL parece de modelo
    elif registro["modelo"] and sopa.h1:
        registro.update({
            "nombre": sopa.h1.get_text(" ", strip=True),
            "precio": precio_visible(sopa),
            "descripcion": meta("description"),
            "imagen_og": meta("og:image"),
            "fuente": "html",
        })
    else:
        return None

    return registro if registro["nombre"] else None


def enlaces_de(sopa, base):
    """Se obtienen todos los enlaces de la página, convertidos a URL absoluta."""
    salida = []
    for a in sopa.find_all("a", href=True):
        salida.append(urljoin(base, a["href"]))
    return salida


# ---------- 4. Bucle principal ----------
productos = []
modelos_vistos = set()
descargadas = bloqueos = sin_producto = lotes = 0
inicio = time.time()


def guardar():
    """Se escribe el CSV con lo reunido hasta ahora, para no perder nada si Colab se desconecta."""
    pd.DataFrame(productos[:META_PRODUCTOS]).to_csv(ARCHIVO, index=False, encoding="utf-8-sig")


while (cola_producto or cola_general) and len(productos) < META_PRODUCTOS and descargadas < MAX_PAGINAS:
    # Armamos un lote; primero van las URLs que parecen de producto
    lote = []
    while len(lote) < LOTE and (cola_producto or cola_general):
        lote.append(cola_producto.popleft() if cola_producto else cola_general.popleft())

    with ThreadPoolExecutor(max_workers=HILOS) as ejecutor:
        resultados = list(ejecutor.map(descargar, lote))

    for url, html, codigo in resultados:
        descargadas += 1
        if html is None:
            if codigo == 403:
                bloqueos += 1
            continue
        sopa = BeautifulSoup(html, "html.parser")
        registro = extraer_producto(sopa, url)
        if registro is None:
            sin_producto += 1
        else:
            clave = registro["modelo"] or url       # evita repetidos (p. ej., versión /business/)
            if clave not in modelos_vistos:
                modelos_vistos.add(clave)
                productos.append(registro)
        for enlace in enlaces_de(sopa, url):
            encolar(enlace)

    lotes += 1
    if lotes % 10 == 0:
        print(f"Páginas: {descargadas} | Productos: {len(productos)} | En cola: {len(cola_producto) + len(cola_general)}")
        guardar()
    if bloqueos >= MAX_BLOQUEOS:
        print("Demasiados bloqueos (403). Se detiene el proceso para no insistir.")
        break

# ---------- 5. Resultado ----------
guardar()
print(f"Listo en {(time.time() - inicio) / 60:.1f} min: {min(len(productos), META_PRODUCTOS)} productos en {ARCHIVO}")
print(f"Páginas descargadas: {descargadas} | sin datos de producto: {sin_producto} | bloqueadas (403): {bloqueos}")
if not productos:
    print("No se encontraron datos de producto. Revisa la sección 'Si sale 0 productos' de la explicación.")

try:
    from google.colab import files
    files.download(ARCHIVO)
except ImportError:
    pass  # fuera de Colab el CSV queda en la carpeta actual

Pausa entre peticiones: 0.5 s
Cola inicial: 1770 URLs de producto y 3948 generales
Páginas: 800 | Productos: 424 | En cola: 5294
Páginas: 1600 | Productos: 698 | En cola: 4508
Páginas: 2400 | Productos: 907 | En cola: 3906
Páginas: 3200 | Productos: 1186 | En cola: 3580
Páginas: 4000 | Productos: 1681 | En cola: 2809
Páginas: 4800 | Productos: 2272 | En cola: 2009
Páginas: 5600 | Productos: 2537 | En cola: 1261
Páginas: 6400 | Productos: 2856 | En cola: 495
Páginas: 7200 | Productos: 2912 | En cola: 817
Páginas: 8000 | Productos: 3082 | En cola: 1082
Páginas: 8800 | Productos: 3273 | En cola: 897
Páginas: 9600 | Productos: 3385 | En cola: 317
Páginas: 10317 | Productos: 3432 | En cola: 3
Listo en 106.2 min: 3432 productos en samsung_mx_productos.csv
Páginas descargadas: 10320 | sin datos de producto: 4821 | bloqueadas (403): 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>